<a href="https://colab.research.google.com/github/AkilaliBeig/AI_Machine_Learning/blob/main/Computer%20Vision/MLS_4_Object_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<h1><center><font size=10>Artificial Intelligence and Machine Learning</center></font></h1>
<h1><center>Computer Vision - Week 4</center></h1>

<center><p float="center">
  <img src="https://images.pexels.com/photos/8090294/pexels-photo-8090294.jpeg?auto=compress&cs=tinysrgb&w=1260&h=750&dpr=1" width="720"/>
</p></center>

<center><font size=6>Face Detection</font></center>

## Importing the necessary libraries

In [ ]:
import cv2
import torch
from PIL import Image
from google.colab.patches import cv2_imshow

# VISUALIZATION
import matplotlib.pyplot as plt # MATPLOTLIB FOR PLOTTING
import seaborn as sns
import pandas as pd
%matplotlib inline

## Data Overview

In [ ]:
image_paths = ['/content/person.jpeg','/content/persons.jpg']

In [ ]:
# Read the images using PIL and opencv libraries
image1 = cv2.imread(image_paths[0])[..., ::-1]  # OpenCV image (BGR to RGB)
image2 = cv2.imread(image_paths[1])[..., ::-1]  # OpenCV image (BGR to RGB)

In [ ]:
import cv2
import matplotlib.pyplot as plt

img_bgr = cv2.imread("person.jpeg")        # BGR
img_rgb = img_bgr[..., ::-1]           # Convert to RGB

plt.subplot(1,2,1)
plt.title("BGR (wrong colors)")
plt.imshow(img_bgr)  # Colors look wrong

plt.subplot(1,2,2)
plt.title("RGB (correct colors)")
plt.imshow(img_rgb)  # Correct colors
plt.show()


In [ ]:
plt.imshow(image1);

In [ ]:
plt.imshow(image2);

## Loading the model

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLOv5s model
model = YOLO("yolov5s.pt")   # or "yolov8n.pt" for YOLOv8 tiny

## Inference on Image

In [ ]:
# create batch/list of images
image_list = [image1, image2]

In [ ]:
# Inferences obtained by passing above images to pretrained model object.
inferences = model(image_list, imgsz = 640)

In [ ]:
# Print the inference using function
# Run inference
# Print results for each image
for i, result in enumerate(inferences):
    print(f"\n--- Image {i+1} ---")
    print(result.boxes.data)    # print detected objects
    # result.save(filename=f"result_{i}.jpg")  # save output if you want

    # 2) To get class names
    for j, box in enumerate(result.boxes):
      cls_id = int(box.cls[0])     # class id
      conf = float(box.conf[0])    # confidence
      xyxy = box.xyxy[0].tolist()  # bounding box coords
      print(f"Class: {model.names[cls_id]}, Conf: {conf:.2f}, BBox: {xyxy}")

      print('-'*130)
      # Show annotated image
      im = result.plot()
      import matplotlib.pyplot as plt
      plt.figure(figsize=(8,6))
      plt.imshow(im)
      plt.axis("off")
      plt.title(f"Detections for Image {i+1}")
      plt.show()


In [ ]:
# we can show image predictions using tensor
result.boxes.data[0:]

In [ ]:
import pandas as pd

# Loop over all inference results
for i, result in enumerate(inferences):
    print(f"\n--- Image {i+1} ---")

    # Convert detections to DataFrame
    df = pd.DataFrame(result.boxes.data.cpu().numpy(),
                      columns=["x1","y1","x2","y2","conf","cls"])

    # Add readable class names
    df["class_name"] = df["cls"].apply(lambda x: model.names[int(x)])

    print(df)   # Print like YOLOv5 .pandas().xyxy[i]


In [ ]:
#Converting xmin,ymin,xmax,ymax to x,y,w,h where w is the width and h is the height.
import pandas as pd

import pandas as pd

# Convert xyxy → xywh
def bbox_transform(xyxy):
    x1, y1, x2, y2 = xyxy
    return x1, y1, x2 - x1, y2 - y1

inferences_list = []

for i, result in enumerate(inferences):   # loop over each image result
    boxes = result.boxes  # ultralytics.yolo.engine.results.Boxes

    # Convert tensor to DataFrame
    df = pd.DataFrame({
        "xmin": boxes.xyxy[:, 0].cpu().numpy(),
        "ymin": boxes.xyxy[:, 1].cpu().numpy(),
        "xmax": boxes.xyxy[:, 2].cpu().numpy(),
        "ymax": boxes.xyxy[:, 3].cpu().numpy(),
        "confidence": boxes.conf.cpu().numpy(),
        "class": boxes.cls.cpu().numpy().astype(int)
    })

    # Add (x, y, w, h)
    df[["x", "y", "w", "h"]] = df.apply(
        lambda row: bbox_transform([row["xmin"], row["ymin"], row["xmax"], row["ymax"]]),
        axis=1, result_type="expand"
    )

    # Add image name
    df["Image_Name"] = image_paths[i]

    # Drop unwanted columns
    df = df.drop(columns=["xmin", "ymin", "xmax", "ymax", "confidence"])

    inferences_list.append(df)

# Aggregate faces only (class = 0)
final_dataframe = []
for df in inferences_list:
    faces = df[df["class"] == 0]
    if not faces.empty:
        grouped = (
            faces.groupby("Image_Name")
                 .agg({"x": list, "y": list, "w": list, "h": list, "class": "size"})
                 .reset_index()[["x", "y", "w", "h", "class", "Image_Name"]]
                 .rename(columns={"class": "Total_Faces"})
        )
        final_dataframe.append(grouped)

# Combine across all images
result = pd.concat(final_dataframe, ignore_index=True)

print(result)


## Inference on Video

In [ ]:
cap = cv2.VideoCapture('/content/car-2165.mp4')

In [ ]:
# Load pretrained YOLOv8 model (change path if you trained your own model)
yolo_model = YOLO("yolov8n.pt")

In [ ]:
while True:
    # Read the next frame from the video
    ret, frame = cap.read()

    if not ret:
        # Break the loop if there are no more frames
        break

    # Apply YOLOv5 to detect objects in the frame
    results = yolo_model(frame)


    # Draw detections directly on the frame
    annotated_frame = results[0].plot()

    # Display the resulting frame
    cv2_imshow(annotated_frame)

    # Wait for a key press to exit
    if cv2.waitKey(1) == ord('q'):
        break


<font size=5 color='blue'>Power Ahead!</font>
___